In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
eeg_eye_state = fetch_ucirepo(id=264) 
  
# data (as pandas dataframes) 
X = eeg_eye_state.data.features 
y = eeg_eye_state.data.targets 
  
# metadata 
print(eeg_eye_state.metadata) 
  
# variable information 
print(eeg_eye_state.variables) 

{'uci_id': 264, 'name': 'EEG Eye State', 'repository_url': 'https://archive.ics.uci.edu/dataset/264/eeg+eye+state', 'data_url': 'https://archive.ics.uci.edu/static/public/264/data.csv', 'abstract': 'The data set consists of 14 EEG values and a value indicating the eye state.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 14980, 'num_features': 14, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['eyeDetection'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2013, 'last_updated': 'Thu Mar 21 2024', 'dataset_doi': '10.24432/C57G7J', 'creators': ['Oliver Roesler'], 'intro_paper': None, 'additional_info': {'summary': "All data is from one continuous EEG measurement with the Emotiv EEG Neuroheadset. The duration of the measurement was 117 seconds. The eye state was detected via a camera during the EEG measuremen

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [3]:
features = X
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.target = y
df.head()

/var/folders/4y/0qtvj2hs6r9_dvmt7n7fbdk40000gn/T/ipykernel_25101/2312714648.py:5: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.target = y


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46


In [4]:
fs = 128

In [5]:
theta_data = []

for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    theta_mask = (freqs >= 4) & (freqs <= 8)

    theta_fft = np.zeros_like(fft_vals)
    theta_fft[theta_mask] = fft_vals[theta_mask]

    theta_signal = np.fft.ifft(theta_fft).real

    theta_data.append(theta_signal)

In [6]:
theta_data = np.array(theta_data)
theta_data = np.transpose(theta_data)

features = theta_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,10.265692,2.809298,4.684134,51.496074,1.629034,17.980050,43.679600,2.982944,6.896455,1.112626,2.478858,2.126970,6.346220,48.844233
1,21.496203,4.213559,7.413818,56.879223,2.185526,9.217122,47.015396,3.025007,14.935382,2.059401,4.432969,3.305647,12.859231,36.175208
2,31.062209,5.446290,9.618600,56.958470,2.554995,-3.355414,45.778020,2.810935,21.724380,2.842159,6.078586,4.264511,18.393672,14.482849
3,37.911996,6.395632,11.084580,51.944154,2.705249,-16.498890,40.289762,2.365957,26.453417,3.371889,7.237767,4.905003,22.289147,-10.066574
4,41.318450,6.965197,11.667693,42.671576,2.627802,-26.834474,31.410308,1.737598,28.571013,3.586103,7.782342,5.159421,24.079962,-30.842937
5,40.975562,7.086340,11.310707,30.468717,2.338093,-31.409410,20.404014,0.990096,27.861841,3.456701,7.649762,4.999373,23.559420,-41.875118
6,37.035956,6.727908,10.050262,16.956225,1.872782,-28.199820,8.750467,0.197125,24.476165,2.993694,6.850849,4.439521,20.808113,-38.870791
7,30.084180,5.902061,8.012964,3.810164,1.284574,-16.471394,-2.072938,-0.566268,18.906744,2.244300,5.468478,3.536111,16.182152,-19.973221
8,21.049973,4.665189,5.400964,-7.476750,0.635338,3.059230,-10.803642,-1.231563,11.916581,1.287539,3.647410,2.380446,10.263100,13.864589
9,11.074457,3.113629,2.468768,-15.799849,-0.011517,28.289209,-16.570003,-1.743592,4.428287,0.225009,1.576649,1.088145,3.776909,58.949362


In [7]:
y = y['eyeDetection']
theta_data_train, theta_data_test, y_train, y_test = train_test_split(theta_data, y, test_size=0.2, random_state=67)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(theta_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [9]:
theta_data_train = scaler.transform(theta_data_train)
theta_data_test = scaler.transform(theta_data_test)

from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [10]:
xgb.fit(theta_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [11]:
y_pred = xgb.predict(theta_data_test)
from sklearn import metrics
metrics.accuracy_score(y_test, y_pred)

0.8137516688918558

In [12]:
notheta_data = []

for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    notheta_mask = (freqs >= 8) | (freqs <= 4)

    notheta_fft = np.zeros_like(fft_vals)
    notheta_fft[notheta_mask] = fft_vals[notheta_mask]

    notheta_signal = np.fft.ifft(notheta_fft).real

    notheta_data.append(notheta_signal)

In [13]:
notheta_data = np.array(notheta_data)
notheta_data = np.transpose(notheta_data)

features = notheta_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4318.964308,4006.420702,4284.545866,4096.713926,4348.630966,4568.169950,4053.240400,4638.047056,4215.153545,4237.347374,4208.801142,4278.383030,4629.553780,4345.005767
1,4303.123797,4000.406441,4286.436182,4091.840777,4339.864474,4577.452878,4050.424604,4635.944993,4195.834618,4224.610599,4203.257031,4276.184353,4619.960769,4347.924792
2,4296.627791,4001.223710,4285.761400,4099.451530,4334.365005,4586.945414,4051.141980,4627.449065,4185.965620,4219.207841,4200.591414,4277.785489,4610.326328,4374.747151
3,4290.808004,4005.394368,4285.325420,4103.955846,4340.884751,4599.058890,4057.150238,4628.404043,4190.986583,4232.008111,4203.532233,4282.784997,4610.020853,4406.476574
4,4284.831550,4004.824803,4280.642307,4108.608424,4345.062198,4613.504474,4064.489692,4625.952402,4182.198987,4240.513897,4205.037658,4283.050579,4608.740038,4429.302937
5,4280.054438,3997.533660,4272.789293,4122.861283,4343.301907,4618.589410,4072.925986,4615.929904,4174.698159,4229.363299,4202.090238,4276.030627,4604.650580,4431.615118
6,4282.454044,3994.302092,4270.459738,4134.833775,4341.717218,4612.819820,4080.989533,4615.702875,4187.833835,4223.676306,4194.179151,4265.300479,4604.321887,4417.330791
7,4295.555820,4000.767939,4270.447036,4139.269836,4342.815426,4599.551394,4089.252938,4615.436268,4186.733256,4228.015700,4190.431522,4263.133889,4605.867848,4400.483221
8,4305.100027,4006.104811,4271.009036,4146.966750,4344.494662,4581.040770,4102.083642,4609.441563,4175.773419,4228.452461,4198.402590,4271.469554,4616.916900,4375.875411
9,4315.075543,4008.166371,4274.451232,4157.849849,4344.111517,4554.270791,4109.390003,4610.463592,4189.931713,4228.494991,4211.243351,4276.861855,4633.663091,4334.380638


In [14]:
notheta_data_train, notheta_data_test, y_train, y_test = train_test_split(notheta_data, y, test_size=0.2, random_state=67)

In [15]:
scaler = StandardScaler()
scaler.fit(notheta_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [16]:
notheta_data_train = scaler.transform(notheta_data_train)
notheta_data_test = scaler.transform(notheta_data_test)

xgb = XGBClassifier()
xgb

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [17]:
xgb.fit(notheta_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [18]:
y_pred = xgb.predict(notheta_data_test)
metrics.accuracy_score(y_test, y_pred)

0.9028704939919893

In [19]:
import random, os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn import metrics

def seed_everything(seed_value):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)

seeds = [67, 69, 41, 523, 0]

results = []

for seed in seeds:
    seed_everything(seed)

    t_train, t_test, y_tr, y_te = train_test_split(
        theta_data, y, test_size=0.2, random_state=seed
    )
    sc_t = StandardScaler().fit(t_train)
    t_train_sc = sc_t.transform(t_train)
    t_test_sc  = sc_t.transform(t_test)

    clf_theta = XGBClassifier(random_state=seed)
    clf_theta.fit(t_train_sc, y_tr)
    acc_theta = metrics.accuracy_score(y_te, clf_theta.predict(t_test_sc))

    nt_train, nt_test, y_tr2, y_te2 = train_test_split(
        notheta_data, y, test_size=0.2, random_state=seed
    )
    sc_nt = StandardScaler().fit(nt_train)
    nt_train_sc = sc_nt.transform(nt_train)
    nt_test_sc  = sc_nt.transform(nt_test)

    clf_notheta = XGBClassifier(random_state=seed)
    clf_notheta.fit(nt_train_sc, y_tr2)
    acc_notheta = metrics.accuracy_score(y_te2, clf_notheta.predict(nt_test_sc))

    results.append((seed, acc_theta, acc_notheta))
    print(f"Seed {seed}  |  Theta: {acc_theta}  |  NoTheta: {acc_notheta}")

Seed 67  |  Theta: 0.8137516688918558  |  NoTheta: 0.9028704939919893
Seed 69  |  Theta: 0.8060747663551402  |  NoTheta: 0.914886515353805
Seed 41  |  Theta: 0.8207610146862483  |  NoTheta: 0.9025367156208278
Seed 523  |  Theta: 0.8144192256341789  |  NoTheta: 0.9088785046728972
Seed 0  |  Theta: 0.8070761014686249  |  NoTheta: 0.9108811748998665
